# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VikramR6/flyrank-ml-internship-assign1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Task type: Classification

The goal is to classify a content page as declining or not declining based on observable page and performance signals. The prediction can then be used to prioritize pages for content review.

In [11]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/VikramR6/flyrank-ml-internship-assign1"
REPO_DIR = "flyrank-ml-internship-assign1"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print(
    "Data exists:",
    os.path.exists("data/raw/content_refresh_anonymized.csv")
)

Working directory: /content/flyrank-ml-internship-assign1/flyrank-ml-internship-assign1
Data exists: True


In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

Target: is_declining

I will use whether the page's trend_direction is "down" as a simple target for this starter exercise. A value of 1 means the page is classified as declining and 0 means it is not declining. This is a proxy for the broader decision of which pages may need content review.

In [13]:
df["is_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Declining pages:", df["is_declining"].sum())
print("Declining rate:", round(df["is_declining"].mean(), 3))

df[[
    "trend_direction",
    "is_declining"
]].head(10)

Declining pages: 16262
Declining rate: 0.542


,trend_direction,is_declining
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


## 3. Success metric

Success metric: F1-score

I will use F1-score because the task has two classes and I care about both identifying declining pages and avoiding too many incorrect alerts. Accuracy alone could hide whether the model is performing well on the declining class. For the eventual review workflow, the usefulness of the highest-priority pages would also matter.

In [14]:
from sklearn.metrics import f1_score

# Baseline: always predict the majority class
majority_class = df["is_declining"].mode()[0]
baseline_pred = np.full(len(df), majority_class)

print("Baseline F1-score:", round(
    f1_score(df["is_declining"], baseline_pred),
    3
))

Baseline F1-score: 0.703


## 4. The unit of analysis, as a real dataframe

Unit of analysis: one content page

Each row represents one content page and contains observable information about that page, such as impressions, average position, CTR, word count, content age, and update recency. The prediction is therefore made at the page level.

In [15]:
unit_cols = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "trend_direction",
    "is_declining"
]

df[unit_cols].head(10)


,impressions_90d,avg_position,ctr,word_count,content_age_days,days_since_last_update,trend_direction,is_declining
0,3803,10.6,0.76,3221.0,187,20,down,1
1,15320,20.3,0.05,2481.0,445,25,down,1
2,12581,36.5,0.09,3515.0,141,20,down,1
3,11751,6.2,0.49,NaN,463,22,stable,0
4,19140,44.0,0.13,2803.0,263,14,down,1
5,3970,8.5,0.03,3080.0,147,20,down,1
6,20,7.0,0.00,3059.0,90,20,down,1
7,1724,21.2,0.06,NaN,445,22,stable,0
8,32574,46.0,0.09,3807.0,90,20,down,1
9,1240,4.9,0.16,NaN,257,104,down,1


## 5. Why ML beats a fixed rule here

A fixed rule such as "stale and visible" is easy to understand, but it uses a small number of manually chosen conditions. ML can help because it can learn combinations and thresholds from several observable signals instead of relying entirely on one hand-written rule. In my earlier experiment, the hand rule performed better at the very top of the ranking, while the decision tree performed slightly better deeper in the list. This mixed result suggests that ML should be evaluated against a simple baseline rather than assumed to be better.

In [16]:
print("Baseline: hand-written rule")
print("ML approach: decision tree using multiple observable signals")
print("Goal: compare both approaches before using the output for content review")


Baseline: hand-written rule
ML approach: decision tree using multiple observable signals
Goal: compare both approaches before using the output for content review


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.